# Morgan Interaction Evaluation

Inspect the saved three-seed Morgan models, then evaluate on the fixed Test split. Requires the trained artifacts produced by the baseline workflow.


## Stage 1

Source: `side_effect_and_baseline_history`, cell 83 (zero-based).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import zipfile
import hashlib
import lightgbm as lgb

ROOT = Path("/content/drive/MyDrive/DDI_V2")
RUN = ROOT / "COMMENT4_REBUILD_20260925_134439"
MORGAN = RUN / "MORGAN_3SEEDS"
TEST = ROOT / "clean_test_split.csv"

assert TEST.exists(), f"Missing: {TEST}"
assert (MORGAN / "config.json").exists()

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=== MORGAN CONFIGURATION ===")
config = json.loads((MORGAN / "config.json").read_text())
print(json.dumps(config, indent=2, ensure_ascii=False))

print("\n=== ALIGNMENT AUDIT ===")
audit_path = RUN / "MORGAN_3SEEDS_ALIGNMENT_AUDIT.json"
assert audit_path.exists(), "Alignment audit is missing."
print(audit_path.read_text())

# Extract seed 44 locally; original Drive ZIP remains unchanged.
zip_path = MORGAN / "MORGAN_CPU_SEED44_COMPLETE.zip"
assert zip_path.exists(), f"Missing: {zip_path}"

local44 = Path("/content/MORGAN44_EVALUATION_INPUT")
local44.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    # Reject archive paths that escape the extraction folder.
    for member in z.infolist():
        target = (local44 / member.filename).resolve()
        assert target.is_relative_to(local44.resolve()), (
            f"Unsafe archive path: {member.filename}"
        )
    z.extractall(local44)

models44 = list(local44.rglob("model.txt"))
assert len(models44) == 1, (
    f"Expected one seed-44 model; found {len(models44)}"
)

model_paths = {
    42: MORGAN / "seed_42" / "model.txt",
    43: MORGAN / "seed_43" / "model.txt",
    44: models44[0],
}

print("\n=== SAVED MODELS — NO TEST METRICS ===")
manifest = {}

for seed, path in model_paths.items():
    assert path.exists(), f"Missing model: {path}"
    completed_path = path.parent / "completed.json"
    assert completed_path.exists(), f"Missing: {completed_path}"

    completed = json.loads(completed_path.read_text())
    assert int(completed["seed"]) == seed

    booster = lgb.Booster(model_file=str(path))

    # Expected dimension of the Morgan pair representation.
    assert booster.num_feature() == 10249
    assert booster.num_model_per_iteration() == 84

    record = {
        "model_path": str(path),
        "model_sha256": sha(path),
        "features": booster.num_feature(),
        "classes": booster.num_model_per_iteration(),
        "saved_iterations": booster.current_iteration(),
        "best_iteration": completed.get("best_iteration"),
    }

    manifest[str(seed)] = record
    print(f"\nSeed {seed}:")
    print(json.dumps(record, indent=2))
    del booster

print("\n=== TEST FILE ===")
print("Path:", TEST)
print("SHA256:", sha(TEST))
print("\n Models located and checked. No test predictions computed.")

## Stage 2

Source: `side_effect_and_baseline_history`, cell 84 (zero-based).


In [ ]:
import json
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import lightgbm as lgb
from rdkit import Chem, rdBase
from rdkit.Chem import rdFingerprintGenerator
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# Uses RUN, MORGAN, TEST, config, model_paths, manifest, sha
# from the previous inspection cell.
assert all(name in globals() for name in [
    "RUN", "MORGAN", "TEST", "config",
    "model_paths", "manifest", "sha"
]), "Run the previous inspection cell first."

OUT = RUN / "MORGAN_3SEEDS_TEST_EVALUATION"
OUT.mkdir(exist_ok=True)

def save_json(path, data):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False))
    tmp.replace(path)

def save_array(path, data):
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, data)
    tmp.replace(path)

def save_csv(path, frame):
    tmp = path.with_name(path.name + ".tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

# Match the molecular-feature and inference environment.
current_versions = {
    "lightgbm": lgb.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "python": platform.python_version(),
    "rdkit": rdBase.rdkitVersion,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
}
for package in ["rdkit", "lightgbm", "numpy", "pandas", "scipy", "sklearn"]:
    assert current_versions[package] == config["versions"][package], (
        f"{package} version differs: "
        f'{current_versions[package]} vs {config["versions"][package]}'
    )

expected_test_hash = (
    "748ee14d51f056025d2e86c324ccdc7d55f"
    "db73f91cdf131bbf847faef793a2f"
)
assert sha(TEST) == expected_test_hash, "Test file changed."

df = pd.read_csv(TEST)
assert len(df) == 28768
assert not df[["Drug1", "Drug2", "label_id"]].isna().any().any()
assert np.array_equal(
    df.label_id.to_numpy(),
    df.label_id.to_numpy(dtype=np.int64)
)
y = df.label_id.to_numpy(dtype=np.int64)
assert np.isin(y, np.arange(84)).all()

# Freeze the evaluation inputs before computing any metrics.
evaluation_spec = {
    "test_sha256": sha(TEST),
    "models": {
        str(seed): {
            "sha256": sha(path),
            "iteration": int(manifest[str(seed)]["best_iteration"]),
        }
        for seed, path in model_paths.items()
    },
    "features": config["features"],
    "class_order": list(range(84)),
    "seeds": [42, 43, 44],
    "prediction_rule": "argmax; smallest class ID on exact tie",
    "macro_f1_labels": list(range(84)),
    "zero_division": 0,
    "sd_ddof": 1,
    "versions": current_versions,
    "selection_on_this_test": False,
    "historical_test_previously_used_for_selection": True,
}

spec_path = OUT / "evaluation_spec.json"
if spec_path.exists():
    assert json.loads(spec_path.read_text()) == evaluation_spec, (
        "Existing evaluation uses different inputs/settings."
    )
else:
    save_json(spec_path, evaluation_spec)

save_csv(
    OUT / "test_rows.csv",
    df[["Drug1", "Drug2", "Label", "label_id"]]
)
save_array(OUT / "class_order.npy", np.arange(84))

# Stateless fingerprints: no fitting on Test.
drugs = pd.concat([
    df.Drug1.astype(str), df.Drug2.astype(str)
]).unique()
drug_to_id = {s: i for i, s in enumerate(drugs)}

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=False
)

fp_rows, fp_cols = [], []
for i, smi in enumerate(tqdm(
    drugs, desc="Test fingerprints", unit="drug"
)):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        raise ValueError(f"Invalid Test SMILES at drug index {i}")
    bits = list(generator.GetFingerprint(mol).GetOnBits())
    fp_rows.extend([i] * len(bits))
    fp_cols.extend(bits)

X_drug = csr_matrix(
    (
        np.ones(len(fp_rows), dtype=np.float32),
        (fp_rows, fp_cols),
    ),
    shape=(len(drugs), 2048),
)

def pair_features(frame):
    a = frame.Drug1.astype(str).map(
        drug_to_id
    ).to_numpy(dtype=np.int64)
    b = frame.Drug2.astype(str).map(
        drug_to_id
    ).to_numpy(dtype=np.int64)

    x1, x2 = X_drug[a], X_drug[b]
    common = x1.multiply(x2)
    only1, only2 = x1-common, x2-common
    only1.eliminate_zeros()
    only2.eliminate_zeros()

    n1 = np.asarray(x1.sum(axis=1)).ravel()
    n2 = np.asarray(x2.sum(axis=1)).ravel()
    nc = np.asarray(common.sum(axis=1)).ravel()
    union = n1+n2-nc

    scalars = np.column_stack([
        n1, n2, nc, union,
        nc/(union+1e-8),
        nc/(n1+1e-8),
        nc/(n2+1e-8),
        n1-n2,
        np.abs(n1-n2),
    ]).astype(np.float32)

    X = hstack(
        [x1, x2, common, only1, only2, csr_matrix(scalars)],
        format="csr",
        dtype=np.float32,
    )
    assert X.shape[1] == 10249
    return X

results = []

for seed in [42, 43, 44]:
    folder = OUT / f"seed_{seed}"
    folder.mkdir(exist_ok=True)

    probs_path = folder / "test_probabilities.npy"
    marker_path = folder / "probabilities_complete.json"

    if probs_path.exists() and marker_path.exists():
        marker = json.loads(marker_path.read_text())
        assert sha(probs_path) == marker["probabilities_sha256"]
        P = np.load(probs_path)
        print(f" Seed {seed}: using saved Test probabilities.")

    else:
        booster = lgb.Booster(model_file=str(model_paths[seed]))
        iteration = evaluation_spec["models"][str(seed)]["iteration"]
        assert booster.num_feature() == 10249
        assert booster.num_model_per_iteration() == 84

        # Save each batch to allow recovery after interruption.
        batch_size = 1024
        P = np.empty((len(df), 84), dtype=np.float32)

        for start in tqdm(
            range(0, len(df), batch_size),
            desc=f"Test seed {seed}",
            unit="batch",
        ):
            end = min(start+batch_size, len(df))
            batch_path = folder / f"batch_{start:06d}_{end:06d}.npy"

            if batch_path.exists():
                batch = np.load(batch_path)
            else:
                batch = np.asarray(
                    booster.predict(
                        pair_features(df.iloc[start:end]),
                        num_iteration=iteration,
                        num_threads=2,
                    ),
                    dtype=np.float32,
                )
                assert batch.shape == (end-start, 84)
                assert np.isfinite(batch).all()
                assert np.allclose(batch.sum(1), 1, atol=1e-5)
                save_array(batch_path, batch)

            assert batch.shape == (end-start, 84)
            assert np.isfinite(batch).all()
            assert batch.min() >= 0 and batch.max() <= 1
            assert np.allclose(batch.sum(1), 1, atol=1e-5)
            P[start:end] = batch

        save_array(probs_path, P)
        save_json(marker_path, {
            "seed": seed,
            "probabilities_sha256": sha(probs_path),
        })
        del booster

    assert P.shape == (len(df), 84)
    assert np.isfinite(P).all()
    assert P.min() >= 0 and P.max() <= 1
    assert np.allclose(P.sum(1), 1, atol=1e-5)

    pred = P.argmax(axis=1)
    save_array(folder / "test_predictions.npy", pred)

    row = {
        "seed": seed,
        "accuracy_pct": 100 * accuracy_score(y, pred),
        "macro_f1_pct": 100 * f1_score(
            y, pred, labels=np.arange(84),
            average="macro", zero_division=0
        ),
        "weighted_f1_pct": 100 * f1_score(
            y, pred, labels=np.arange(84),
            average="weighted", zero_division=0
        ),
        "best_iteration":
            evaluation_spec["models"][str(seed)]["iteration"],
    }
    save_json(folder / "test_metrics.json", row)
    results.append(row)

all_results = pd.DataFrame(results)
save_csv(OUT / "all_seed_test_results.csv", all_results)

summary_rows = []
for metric in ["accuracy_pct", "macro_f1_pct", "weighted_f1_pct"]:
    values = all_results[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    summary_rows.append({
        "metric": metric,
        "n_runs": 3,
        "mean_pct": mean,
        "sd_percentage_points": sd,
        "mean_plus_minus_sd": f"{mean:.2f} ± {sd:.2f}",
    })

summary = pd.DataFrame(summary_rows)
save_csv(OUT / "three_seed_test_mean_sd.csv", summary)

print("\n=== INDIVIDUAL TEST RESULTS ===")
display(all_results)

print("\n=== MEAN ± SAMPLE SD — 3 SEEDS ===")
display(summary)

print("\n Saved:", OUT)
print("No training or model selection was performed in this cell.")